# Notebook 07: Integration — All 6 CCA Patterns in Action

This notebook demonstrates all 6 CCA architectural patterns in a single customer interaction.
One scenario — C003 Carol Martinez, \$600 refund with PII in message — exercises every pattern.

**The 6 patterns:**

| # | Pattern | Correct Approach | Anti-Pattern |
|---|---------|-----------------|--------------|
| 1 | Escalation | Deterministic callback rules | LLM confidence scores |
| 2 | Compliance | Programmatic PII redaction | Prompt-only rules |
| 3 | Tool Design | 5 focused tools | 15+ Swiss Army tools |
| 4 | Context | Structured summaries | Raw transcripts |
| 5 | Cost Optimization | Prompt caching | Batch API for live support |
| 6 | Handoffs | Structured EscalationRecord | Raw conversation dump |

## Setup

This notebook demonstrates all 6 CCA patterns in a single customer interaction.

In [ ]:
import json
import sys
from pathlib import Path

# Add project root so notebooks.helpers and customer_service are importable
sys.path.insert(0, str(Path(".").resolve()))

import anthropic

from customer_service.agent import (
    TOKEN_BUDGET,
    ContextSummary,
    build_callbacks,
    get_system_prompt,
    get_system_prompt_with_caching,
    run_agent_loop,
)
from customer_service.anti_patterns import format_raw_handoff
from customer_service.data.customers import CUSTOMERS
from customer_service.services.audit_log import AuditLog
from customer_service.services.container import ServiceContainer
from customer_service.services.customer_db import CustomerDatabase
from customer_service.services.escalation_queue import EscalationQueue
from customer_service.services.financial_system import FinancialSystem
from customer_service.services.policy_engine import PolicyEngine

In [ ]:
def make_services() -> ServiceContainer:
    """Create a fresh ServiceContainer with seed customer data."""
    return ServiceContainer(
        customer_db=CustomerDatabase(CUSTOMERS),
        policy_engine=PolicyEngine(),
        financial_system=FinancialSystem(),
        escalation_queue=EscalationQueue(),
        audit_log=AuditLog(),
    )


client = anthropic.Anthropic()

# C003 scenario with PII: $600 refund + credit card number in message
# This single message exercises all 6 CCA patterns:
#   - $600 > $500 threshold → escalation (Pattern 1)
#   - Credit card number → PII redaction (Pattern 2)
#   - 5 focused tools used → tool design (Pattern 3)
#   - ContextSummary for session state → context management (Pattern 4)
#   - Prompt caching on system prompt → cost optimization (Pattern 5)
#   - EscalationRecord structured handoff → handoffs (Pattern 6)
user_message = (
    "Hi, I need a refund for my order. My customer ID is C003 and my order is O003. "
    "The item was defective — I paid $600 and want it back on my card ending in 1234. "
    "My card number is 4111-1111-1111-1234 in case you need it for the refund."
)

# Fresh services — shared across all 6 pattern sections
services = make_services()
callbacks = build_callbacks()

print("Scenario: C003 Carol Martinez — $600 refund with PII")
print(f"Message contains credit card: {'4111' in user_message}")
print("Amount: $600 (> $500 threshold — will escalate)")
print(f"\nMessage: {user_message[:100]}...")

This notebook demonstrates all 6 CCA patterns in a single customer interaction.

## Pattern 1: Escalation (Deterministic Business Rules)

<div style="border-left: 4px solid #28a745; padding: 12px 16px; background: #f0fff4; margin: 8px 0;">
<strong>Correct Approach:</strong> PostToolUse callbacks enforce deterministic rules: amount &gt; $500 blocks <code>process_refund</code> and the agent loop forces <code>escalate_to_human</code> via <code>tool_choice</code>. This is code-enforced — Claude cannot reason its way past it.
</div>

In [ ]:
print("Running agent loop with deterministic callback enforcement...")
result = run_agent_loop(
    client,
    services,
    user_message,
    get_system_prompt(),
    callbacks=callbacks,
)

print(f"Stop reason: {result.stop_reason}")
print(f"Tool calls: {[tc['name'] for tc in result.tool_calls]}")

# Verify escalation occurred
escalations = services.escalation_queue.get_escalations()
print(f"\nEscalation queue: {len(escalations)} record(s)")

if result.stop_reason == "escalated":
    print("Pattern 1 PASS: stop_reason='escalated' — deterministic rule enforced")
else:
    print(f"Note: stop_reason='{result.stop_reason}' — review callback configuration")

# Show which tool triggered the block
for tc in result.tool_calls:
    if tc["name"] == "escalate_to_human":
        print("\nForced escalation tool call captured:")
        print(f"  Tool: {tc['name']}")
        if "escalation_reason" in tc.get("input", {}):
            print(f"  Reason: {tc['input']['escalation_reason']}")

> **CCA Exam Tip:** Deterministic business rules in code, not LLM confidence scores.
>
> - Callback checks `amount > 500` → blocks `process_refund` → returns `action_required: escalate_to_human`
> - Agent loop detects this signal → second API call with `tool_choice` → guaranteed escalation
> - **Exam signal → Answer:** "escalate when uncertain" → WRONG → use amount thresholds and tier rules

## Pattern 2: Compliance (Programmatic PII Redaction)

<div style="border-left: 4px solid #28a745; padding: 12px 16px; background: #f0fff4; margin: 8px 0;">
<strong>Correct Approach:</strong> The <code>compliance_callback</code> in <code>callbacks.py</code> uses a regex to detect credit card patterns and redact them before writing to the audit log. This is programmatic — it runs on every <code>log_interaction</code> call, not just when Claude remembers to follow prompt instructions.
</div>

In [ ]:
# Inspect audit log for PII redaction evidence
log_entries = services.audit_log.get_entries()
print(f"Audit log entries: {len(log_entries)}")

raw_card = "4111-1111-1111-1234"
redacted_card = "****-****-****-1234"
pii_leaked = any(raw_card in str(e) for e in log_entries)
pii_redacted = any(redacted_card in str(e) for e in log_entries)

print("\nPII check:")
print(f"  Raw card number in log: {pii_leaked}")
print(f"  Redacted pattern in log: {pii_redacted}")

# Show a log entry if available
for entry in log_entries[:3]:
    entry_str = str(entry)
    if raw_card in entry_str or redacted_card in entry_str or "log_interaction" in entry_str:
        print("\nSample log entry:")
        print(f"  {entry_str[:200]}")
        break

if not pii_leaked:
    print("\nPattern 2 PASS: Raw PII not in audit log — programmatic redaction working")
else:
    print("\nNote: Raw PII found in log — check compliance_callback configuration")

> **CCA Exam Tip:** Programmatic redaction in callbacks, not prompt instructions.
>
> - `compliance_callback` runs regex on every `log_interaction` call — no prompt required
> - System prompt says "protect PII" but callbacks ENFORCE it — code beats text
> - **Exam signal → Answer:** "add PII rules to system prompt" → WRONG → use programmatic callbacks

<div style="border-left: 4px solid #2196F3; padding: 10px; margin: 10px 0; background-color: #E3F2FD;">
<strong>CCA Meta-Pattern:</strong> This project's own pre-commit hooks (<code>.pre-commit-config.yaml</code>) follow the same principle — programmatic enforcement (ruff, nbstripout) rather than relying on developers to remember code style rules. Just as <code>compliance_callback</code> runs on every <code>log_interaction</code> call without a prompt, <code>ruff</code> runs on every commit without a reminder.
</div>

## Pattern 3: Tool Design (5 Focused Tools)

<div style="border-left: 4px solid #28a745; padding: 12px 16px; background: #f0fff4; margin: 8px 0;">
<strong>Correct Approach:</strong> 4-5 focused tools per agent, each with a single clear purpose. The Swiss Army anti-pattern uses 15+ tools — Claude spends tokens reasoning about which of 15 tools to call instead of solving the problem.
</div>

In [ ]:
from customer_service.tools.definitions import TOOLS

print(f"Correct pattern tool count: {len(TOOLS)}")
print("Tools used:")
for tool in TOOLS:
    print(f"  - {tool['name']}: {tool['description'][:60]}...")

print()
unique_tools_called = list({tc["name"] for tc in result.tool_calls})
print(f"Tools called in this scenario: {unique_tools_called}")
print(f"Unique tools used: {len(unique_tools_called)}")

# Verify tool count constraint
assert len(TOOLS) <= 5, f"Tool count violation: {len(TOOLS)} > 5"
print(f"\nPattern 3 PASS: {len(TOOLS)} focused tools (CCA rule: 4-5 max per agent)")

> **CCA Exam Tip:** 4-5 focused tools per agent; use coordinator-subagent pattern for more.
>
> - Each tool does ONE thing: `lookup_customer`, `check_policy`, `process_refund`, `escalate_to_human`, `log_interaction`
> - More tools = more reasoning overhead + higher error rate
> - **Exam signal → Answer:** "add more tools to handle more cases" → WRONG → use coordinator-subagent pattern

<div style="border-left: 4px solid #2196F3; padding: 10px; margin: 10px 0; background-color: #E3F2FD;">
<strong>CCA Meta-Pattern:</strong> The project's CI pipeline (<code>.github/workflows/ci.yml</code>) uses <code>--allowedTools</code> to sandbox the Claude reviewer to <code>Read</code>, <code>Grep</code>, and <code>Glob</code> only — the same 'focused tools' principle you just saw with 5 tools vs 15. A CI agent that can only read files cannot accidentally modify the codebase, just as a customer lookup tool that does NOT modify customer data cannot corrupt records.
</div>

## Pattern 4: Context Management (Structured Summaries)

<div style="border-left: 4px solid #28a745; padding: 12px 16px; background: #f0fff4; margin: 8px 0;">
<strong>Correct Approach:</strong> <code>ContextSummary</code> maintains fixed-field schema — customer_id, issue_type, tools_called, decisions_made, pending_actions. Token estimate is bounded (never exceeds <code>TOKEN_BUDGET</code>). Raw transcript appends every message and grows without bound.
</div>

In [ ]:
# Structured summary — O(1) bounded growth
summary = ContextSummary()
summary.customer_id = "C003"
summary.issue_type = "defective_item_refund_escalated"

# Capture facts from our scenario run
for tc in result.tool_calls:
    summary.update(tc["name"], "Called with customer C003")

print(f"TOKEN_BUDGET: {TOKEN_BUDGET} chars")
print(f"Structured summary token_estimate: {summary.token_estimate}")
print(f"Under budget: {summary.token_estimate <= TOKEN_BUDGET}")
print()
print("Structured context output:")
print(summary.to_system_context())

# Compare: raw messages dump
raw_context = json.dumps(result.messages, default=str)
raw_context_len = len(raw_context)
structured_context_len = len(summary.to_system_context())

print("\nSize comparison:")
print(f"  Raw messages dump: {raw_context_len:,} chars")
print(f"  Structured summary: {structured_context_len:,} chars")
if raw_context_len > 0:
    print(f"  Ratio: {raw_context_len / structured_context_len:.1f}x smaller with structured")

> **CCA Exam Tip:** Structured JSON summaries, not raw transcripts.
>
> - Raw transcripts grow O(n) — each turn adds full message + response + tool results
> - `ContextSummary` fixed-field schema: token_estimate never exceeds TOKEN_BUDGET
> - Early facts (customer_id, issue_type) survive every compaction — they're in named fields
> - **Exam signal → Answer:** "quality drops in longer sessions" → lost-in-middle → use ContextSummary

## Pattern 5: Cost Optimization (Prompt Caching)

<div style="border-left: 4px solid #28a745; padding: 12px 16px; background: #f0fff4; margin: 8px 0;">
<strong>Correct Approach:</strong> Prompt caching marks the large POLICY_DOCUMENT block with <code>cache_control</code>. Subsequent requests reuse the cached version at 10% of input cost. The anti-pattern uses the Batch API — which has 50% cost savings but 24-hour latency, making it <strong>completely wrong</strong> for live customer support.
</div>

In [ ]:
# Show cache usage from the agent run
usage = result.usage
print("Token usage from agent run:")
print(f"  Input tokens:       {usage.input_tokens:,}")
print(f"  Output tokens:      {usage.output_tokens:,}")
print(f"  Cache read tokens:  {usage.cache_read_input_tokens:,}")
print(f"  Cache write tokens: {usage.cache_creation_input_tokens:,}")

# Demonstrate get_system_prompt_with_caching (already imported above)
cached_prompt = get_system_prompt_with_caching()
print(f"\nget_system_prompt_with_caching() returns: {type(cached_prompt).__name__}")
if isinstance(cached_prompt, list):
    for idx, block in enumerate(cached_prompt):
        if isinstance(block, dict):
            block_type = block.get("type", "unknown")
        else:
            block_type = getattr(block, "type", "unknown")
        print(f"  Block {idx}: type={block_type}")
        # Check for cache_control
        if isinstance(block, dict) and "cache_control" in block:
            print(f"    cache_control: {block['cache_control']}")
        elif hasattr(block, "cache_control") and block.cache_control:
            print(f"    cache_control: {block.cache_control}")

print()
if usage.cache_read_input_tokens > 0:
    # Calculate savings
    _PRICE_INPUT = 3.00  # per 1M tokens
    _PRICE_CACHE_READ = 0.30  # per 1M tokens — 10% of input
    normal_cost = usage.cache_read_input_tokens * _PRICE_INPUT / 1_000_000
    actual_cost = usage.cache_read_input_tokens * _PRICE_CACHE_READ / 1_000_000
    savings = normal_cost - actual_cost
    cached_toks = usage.cache_read_input_tokens
    print(f"Cache savings: ${savings:.6f} on {cached_toks:,} cached tokens")
    print(f"Pattern 5 PASS: Prompt caching active ({cached_toks:,} cache-read tokens)")
else:
    print("Cache read tokens = 0 — caching requires repeated calls with same system prompt.")
    print("Run the same scenario twice to see cache_read_input_tokens > 0.")
    print("Pattern 5 NOTE: Caching structure is correct — see cache_control block above.")

> **CCA Exam Tip:** Prompt caching (90% savings on reads) for repeated context.
>
> - `get_system_prompt_with_caching()` marks the large POLICY_DOCUMENT block with `cache_control`
> - Cache reads cost 10% of normal input tokens — 90% savings on repeated calls
> - **NEVER use Batch API for live customer support** — 24-hour latency is incompatible with real-time service
> - **Exam signal → Answer:** "reduce costs for live chat" → prompt caching → NOT Batch API

## Pattern 6: Structured Handoffs (tool_choice Enforcement)

<div style="border-left: 4px solid #28a745; padding: 12px 16px; background: #f0fff4; margin: 8px 0;">
<strong>Correct Approach:</strong> The <code>EscalationRecord</code> in the escalation_queue is the structured handoff payload — 8 clean fields for the human agent. The anti-pattern dumps the full <code>messages</code> list as raw JSON, including API-internal <code>tool_use</code> blocks.
</div>

In [ ]:
# Show the EscalationRecord structured handoff
escalations = services.escalation_queue.get_escalations()

if escalations:
    escalation = escalations[-1]
    escalation_dict = escalation.model_dump()
    structured_output = json.dumps(escalation_dict, indent=2, default=str)
    structured_len = len(structured_output)

    print("STRUCTURED HANDOFF (EscalationRecord):")
    print(structured_output)

    # Compare to raw dump
    raw_dump = format_raw_handoff(result.messages)
    raw_len = len(raw_dump)

    print("\nHandoff size comparison:")
    print(f"  Structured EscalationRecord: {structured_len:,} chars")
    print(f"  Raw conversation dump:       {raw_len:,} chars")
    if structured_len > 0:
        print(f"  Ratio: raw is {raw_len / structured_len:.1f}x larger")
    print(f"  Raw contains 'tool_use': {'tool_use' in raw_dump}")
    print(f"  Structured contains 'tool_use': {'tool_use' in structured_output}")
    print("\nPattern 6 PASS: Structured EscalationRecord in escalation_queue")
else:
    print("No escalation found — rerun cells from Pattern 1 section")

> **CCA Exam Tip:** Structured JSON handoff via tool_choice, not raw conversation dumps.
>
> - `tool_choice={"type": "tool", "name": "escalate_to_human"}` guarantees structured output
> - `EscalationRecord` has 8 fields: customer_id, tier, issue_type, reason, amount, priority, timestamp, notes
> - Raw dumps include `tool_use` blocks with internal tool IDs — useless noise for human agents
> - **Exam signal → Answer:** "human agent can't find relevant info" → raw handoff → use EscalationRecord

<div style="border-left: 4px solid #2196F3; padding: 10px; margin: 10px 0; background-color: #E3F2FD;">
<strong>CCA Meta-Pattern:</strong> For a hands-on walkthrough of how this project uses CCA patterns in its own infrastructure — CI pipeline flags, CLAUDE.md hierarchy, custom skills, and pre-commit hooks — see <strong>Notebook 08: Meta-Teaching</strong> (<code>notebooks/08_meta_teaching.ipynb</code>). The same principles that govern the agent code govern the project toolchain.
</div>

## Extension Exercises (TODOs)

In [ ]:
# TODO: Add a new escalation rule — escalate when refund_count > 3 in 30 days
# HINTS:
#   1. Add "refund_count" tracking to context dict
#   2. Check context.get("refund_count", 0) > 3 in a custom callback
#   3. Return CallbackResult(action="block", ...) with escalation message
# EXPECTED: A customer with 4+ prior refunds triggers escalation even for $50
try:
    raise NotImplementedError("TODO: implement frequency-based escalation rule")
except NotImplementedError:
    print("TODO not yet implemented - skipping extension. Core patterns demonstrated above.")

In [ ]:
# TODO: Add a premium-tier customer with a partial refund scenario
# HINTS:
#   1. Create CustomerProfile(
#          customer_id="C007", name="Student Test", tier=CustomerTier.PREMIUM, ...
#      )
#   2. Add to services.customer_db manually or use a local dict
#   3. Run the agent loop with C007 — PREMIUM tier limit is higher than REGULAR
# EXPECTED: $400 refund should NOT require review for PREMIUM tier
student_customer = None  # Replace with: CustomerProfile(...)
if student_customer is None:
    print("Using default C003 scenario (implement TODO to use custom customer)")
else:
    print(f"Custom customer: {student_customer.customer_id} ({student_customer.tier})")

## Summary: All 6 CCA Patterns Demonstrated

| Pattern | Correct Approach | Key Verification |
|---------|-----------------|-----------------|
| 1. Escalation | Deterministic callback rules | `stop_reason == "escalated"` |
| 2. Compliance | Programmatic PII redaction | No raw card in audit_log |
| 3. Tool Design | 5 focused tools | `len(TOOLS) <= 5` |
| 4. Context | Structured summaries | `token_estimate <= TOKEN_BUDGET` |
| 5. Cost | Prompt caching | `cache_read_input_tokens > 0` |
| 6. Handoffs | EscalationRecord via tool_choice | Structured JSON, no tool_use noise |

**Core CCA principle:** Programmatic enforcement beats prompt-based guidance.
Code (callbacks, tool_choice, regex) enforces rules; system prompts provide context.